In [1]:
## Restructured model with germany data

In [2]:
import pickle
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data = pickle.load(file)['contacts']

In [3]:
data

,id,age_part,sex_part,occ_part,edu_part,hh_size,class_size,work_contacts_nr,age_cnt,sex_cnt,y
0,846,10.0,F,5.0,2.0,4,26.0,NaN,43.0,M,3.0
1,846,10.0,F,5.0,2.0,4,26.0,NaN,70.0,F,2.0
2,846,10.0,F,5.0,2.0,4,26.0,NaN,68.0,M,2.0
3,846,10.0,F,5.0,2.0,4,26.0,NaN,11.0,F,1.0
4,846,10.0,F,5.0,2.0,4,26.0,NaN,13.0,F,2.0
...,...,...,...,...,...,...,...,...,...,...,...
10648,2185,70.0,M,2.0,8.0,2,NaN,NaN,57.0,M,1.0
10649,2185,70.0,M,2.0,8.0,2,NaN,NaN,57.0,M,1.0
10650,2185,70.0,M,2.0,8.0,2,NaN,NaN,61.0,M,1.0
10651,2185,70.0,M,2.0,8.0,2,NaN,NaN,32.0,M,1.0


In [4]:
from cntmosaic.dataloader.restru_loaders import MergedLoader, RawLoader

In [5]:
import pandas as pd
import numpy as np

In [6]:
import xarray as xr

In [7]:
from cntmosaic.dataloader import CoordToColumns

In [8]:
!pwd

/home/yiminglin/Downloads/cntmosaic/cntmosaic/tutorial


In [9]:
c = pd.read_csv('2008_Mossong_POLYMOD_contact_common.csv')

In [10]:
p = pd.read_csv('2008_Mossong_POLYMOD_participant_common.csv')

In [11]:
p

,part_id,hh_id,part_age,part_gender
0,1,Mo08HH1,8.0,F
1,2,Mo08HH2,6.0,M
2,3,Mo08HH3,0.0,F
3,4,Mo08HH4,1.0,M
4,5,Mo08HH5,2.0,M
...,...,...,...,...
7285,8000,Mo08HH8000,26.0,M
7286,8001,Mo08HH8001,14.0,F
7287,50036,Mo08HH50036,45.0,F
7288,50037,Mo08HH50037,54.0,F


In [12]:
c

,cont_id,part_id,cnt_age_exact,cnt_age_est_min,cnt_age_est_max,cnt_gender,cnt_home,cnt_work,cnt_school,cnt_transport,cnt_leisure,cnt_otherplace,frequency_multi,phys_contact,duration_multi
0,1,1,42.0,NaN,NaN,F,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,4.0
1,2,1,9.0,NaN,NaN,F,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,5.0
2,3,1,NaN,40.0,45.0,F,0.0,0.0,0.0,0.0,1.0,0.0,3.0,1.0,4.0
3,4,1,8.0,NaN,NaN,F,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,5.0
4,5,1,NaN,28.0,30.0,F,0.0,0.0,1.0,0.0,0.0,0.0,2.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97899,97900,8001,11.0,NaN,NaN,F,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,5.0
97900,97901,8001,3.0,NaN,NaN,M,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,5.0
97901,97902,8001,43.0,NaN,NaN,F,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,5.0
97902,97903,8001,33.0,NaN,NaN,F,0.0,0.0,0.0,0.0,0.0,1.0,4.0,1.0,3.0


In [13]:
mapp = CoordToColumns(age_part='part_age', age_cnt='cnt_age_exact', id_var='part_id')

In [14]:
data = RawLoader(p, c, mapp)

In [15]:
#d = MergedLoader(data, mapp)

In [16]:
data.raw_df = data.raw_df.groupby(['part_age', 'part_id', 'cnt_age_exact']).sum()['y'].reset_index()

In [17]:
data.load()

In [18]:
data.ds

<xarray.Dataset> Size: 464MB
Dimensions:        (part_id: 5805, part_age: 100, cnt_age_exact: 100)
Coordinates:
  * part_id        (part_id) int64 46kB 1 2 3 4 5 6 ... 7997 7998 7999 8000 8001
  * part_age       (part_age) int64 800B 0 1 2 3 4 5 6 ... 93 94 95 96 97 98 99
  * cnt_age_exact  (cnt_age_exact) int64 800B 0 1 2 3 4 5 ... 94 95 96 97 98 99
Data variables:
    y              (part_id, part_age, cnt_age_exact) float64 464MB nan ... nan

In [19]:
from cntmosaic.models.restr_BRCfine import restr_BRCfine

In [20]:
model = restr_BRCfine(data)

new model instantiated, please check default hyperparameters


In [21]:
model.compile()

model compiled, ready for sampling


In [22]:
import jax
model.run_inference_mcmc(jax.random.PRNGKey(0),
    num_samples = 10,
    num_warmup = 5,
    num_chains = 2)

/home/yiminglin/Downloads/cntmosaic/cntmosaic/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 2 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(2)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 15/15 [00:22<00:00,  1.51s/it, 511 steps of size 8.76e-03. acc. prob=0.60]


Number of divergences: 10


In [15]:
post=model.mcmc.get_samples()

In [19]:
post['log_cint']

Array([[[ -5.3639345,  -5.556456 ,  -5.777896 , ...,  -9.048573 ,
          -9.17355  ,  -9.287203 ],
        [ -5.556456 ,  -5.242554 ,  -5.464059 , ...,  -9.1304   ,
          -9.270456 ,  -9.401534 ],
        [ -5.777896 ,  -5.464059 ,  -5.105899 , ...,  -9.220484 ,
          -9.372832 ,  -9.515657 ],
        ...,
        [ -9.048573 ,  -9.1304   ,  -9.220484 , ...,  -9.5451355,
          -9.617667 ,  -9.67491  ],
        [ -9.17355  ,  -9.270456 ,  -9.372832 , ...,  -9.617667 ,
          -9.719809 ,  -9.773178 ],
        [ -9.287203 ,  -9.401534 ,  -9.515657 , ...,  -9.67491  ,
          -9.773178 ,  -9.845547 ]],

       [[ -6.086517 ,  -6.1599045,  -6.252522 , ..., -10.643705 ,
         -10.795526 , -10.909746 ],
        [ -6.1599045,  -5.785299 ,  -5.894202 , ..., -10.559336 ,
         -10.715267 , -10.828617 ],
        [ -6.252522 ,  -5.894202 ,  -5.518101 , ..., -10.457129 ,
         -10.613918 , -10.727275 ],
        ...,
        [-10.643705 , -10.559336 , -10.457129 , ...,  

In [ ]:
from cntmosaic.sim import ModelEvaluatorMCMC

In [ ]:
e = ModelEvaluatorMCMC(model)

In [ ]:
e.post = model.mcmc.get_samples(group_by_chain=True)
#e.get_posterior() extra dimension for different chains

In [ ]:
import numpy as np

In [ ]:
log_rate = e.post['log_rate']
post_cint = {}
for name, site in e.post.items():
    if 'log_delta' in name:
        var = name.split('/')[0]
        cat = e.model.data[var].cat.categories
        post_cint[var] = {
            cat[i]: np.exp(log_rate[:, None, :, :] + site + e.model._precompute.log_P[None, None, :, :])[:, i, :, :]
            for i in range(len(cat))
        }

In [ ]:
e.post.keys()

In [ ]:
e.post['inv_disp'].shape

In [ ]:
import numpyro
import jax.numpy as jnp
from numpyro import distributions as dist

def guide():
    # --- beta0: scalar Normal(0, 10) ---
    beta0_loc = numpyro.param("beta0_loc", jnp.zeros(()))
    beta0_scale = numpyro.param("beta0_scale", jnp.ones(()), constraint=dist.constraints.positive)
    numpyro.sample("beta0", dist.Normal(beta0_loc, beta0_scale))

    # --- alpha: scalar InverseGamma(5, 5) ---
    alpha_conc = numpyro.param("alpha_conc", jnp.ones(()), constraint=dist.constraints.positive)
    alpha_rate = numpyro.param("alpha_rate", jnp.ones(()), constraint=dist.constraints.positive)
    numpyro.sample("baseline", dist.InverseGamma(alpha_conc, alpha_rate))

    # --- rho: vector of 2 InverseGamma(5, 5) ---
    rho_conc = numpyro.param("rho_conc", jnp.ones(2), constraint=dist.constraints.positive)
    rho_rate = numpyro.param("rho_rate", jnp.ones(2), constraint=dist.constraints.positive)
    numpyro.sample("rho", dist.InverseGamma(rho_conc, rho_rate))

    inv_disp_rate = numpyro.param("inv_disp_rate", jnp.ones(len(out)), constraint=dist.constraints.positive)
    numpyro.sample("inv_disp", dist.Exponential(inv_disp_rate))


In [ ]:
model.run_inference_svi(jax.random.PRNGKey(0),guide, num_steps = 1000)

In [ ]:
from cntmosaic.sim import ModelEvaluatorSVI
svi = ModelEvaluatorSVI(model)

In [ ]:
svi.get_pred_cint()

In [ ]:
svi.pred_cint